State: 
0 = Sạch, 
1 = Bụi, 
2 = Robot, 
3 = Chướng ngại vật

In [49]:
import tkinter as tk
from tkinter import ttk, scrolledtext
import random
import time
from collections import deque
import heapq

In [50]:
class Node:
    name_counter = 0

    @classmethod
    def reset_counter(cls):
        cls.name_counter = 0

    @classmethod
    def get_next_name(cls):
        # Tự động sinh tên Node: A, B, C... Z, AA, AB...
        n = cls.name_counter
        cls.name_counter += 1
        name = ""
        while True:
            name = chr(n % 26 + 65) + name
            n = n // 26 - 1
            if n < 0: break
        return name

    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action
        self.name = Node.get_next_name()
        # Cost bằng cost của cha + 1 (Root có cost = 0)
        self.cost = 0 if parent is None else parent.cost + 1

In [51]:
def copy_matrix(matrix):
    return [row[:] for row in matrix]

In [52]:
def find_robot(matrix):
    rows = len(matrix)
    cols = len(matrix[0])
    
    for i in range(rows):
        for j in range(cols):
            if matrix[i][j] == 2:
                return [i, j]
            
    return None

In [53]:
def goal_test(state):
    for row in state:
        if 1 in row:
            return False
    return True

In [54]:
def actions(state):
    x, y = find_robot(state)
    move = []
    
    rows = len(state)
    cols = len(state[0])
    
    if x > 0 and state[x-1][y] != 3:
        move.append("UP")
    if x < rows - 1 and state[x+1][y] != 3:
        move.append("DOWN")
    if y > 0 and state[x][y-1] != 3:
        move.append("LEFT")
    if y < cols - 1 and state[x][y+1] != 3:
        move.append("RIGHT")
        
    return move

In [55]:
def result(state, action):
    new_state = copy_matrix(state)

    x, y = find_robot(new_state)
    new_state[x][y] = 0

    nx = x
    ny = y

    if action == "UP":
        nx -= 1
    elif action == "DOWN":
        nx += 1
    elif action == "LEFT":
        ny -= 1
    elif action == "RIGHT":
        ny += 1

    new_state[nx][ny] = 2
    return new_state

In [56]:
def child_node(node, action):
    new_state = result(node.state, action)
    return Node(new_state, node, action)

In [57]:
def solution(node):
    path = []

    while node.parent is not None:
        path.append(node.action)
        node = node.parent
    path.reverse()

    return path

In [58]:
def move_robot(matrix, action):
    x, y = find_robot(matrix)

    matrix[x][y] = 0

    if action == "UP":
        x -= 1
    elif action == "DOWN":
        x += 1
    elif action == "LEFT":
        y -= 1
    elif action == "RIGHT":
        y += 1

    matrix[x][y] = 2

In [59]:
def count_dust(matrix):
    count = 0

    for row in matrix:
        count += row.count(1)

    return count

In [60]:
def breadth_first_search_1(initial_state, gui):
    node = Node(initial_state)

    if goal_test(node.state):
        return solution(node)

    frontier = deque()
    frontier.append(node)
    reached = []

    while len(frontier) > 0:
        node = frontier.popleft() # Lấy node ở đầu frontier ra (Queue: FIFO)
        reached.append(node.state)
        
        if goal_test(node.state):
            return solution(node)

        for action in actions(node.state):
            child = child_node(node, action)
            
            in_reached = False
            for s in reached:
                if s == child.state:
                    in_reached = True

            in_frontier = False
            for n in frontier:
                if n.state == child.state:
                    in_frontier = True

            if not in_reached and not in_frontier:
                frontier.append(child)
                
        gui.log_step(node, frontier, reached, "Reached") # Gọi hàm log_step để cập nhật giao diện sau mỗi bước
     
    return None

In [61]:
def breadth_first_search_2(initial_state, gui):
    node = Node(initial_state)

    if goal_test(node.state):
        return solution(node)

    frontier = deque()
    frontier.append(node)
    explored = []

    while len(frontier) > 0:
        node = frontier.popleft() # Lấy node ở đầu frontier ra (Queue: FIFO)
        explored.append(node.state)

        for action in actions(node.state):
            child = child_node(node, action)

            in_explored = False
            for s in explored:
                if s == child.state:
                    in_explored = True

            in_frontier = False
            for n in frontier:
                if n.state == child.state:
                    in_frontier = True

            if not in_explored and not in_frontier:
                if goal_test(child.state):
                    return solution(child)

                frontier.append(child)
        
        gui.log_step(node, frontier, explored, "Explored") # Gọi hàm log_step để cập nhật giao diện sau mỗi bước
                
    return None

In [62]:
def depth_first_search_1(initial_state, gui):
    node = Node(initial_state)

    if goal_test(node.state):
        return solution(node)

    frontier = deque()
    frontier.append(node)
    reached = []

    while len(frontier) > 0:
        node = frontier.pop() # Lấy node ở cuối frontier ra (Stack: LIFO)
        reached.append(node.state)

        if goal_test(node.state):
            return solution(node)

        for action in actions(node.state):
            child = child_node(node, action)
            
            in_reached = False
            for s in reached:
                if s == child.state:
                    in_reached = True

            in_frontier = False
            for n in frontier:
                if n.state == child.state:
                    in_frontier = True

            if not in_reached and not in_frontier:
                frontier.append(child)
                
        gui.log_step(node, frontier, reached, "Reached") # Gọi hàm log_step để cập nhật giao diện sau mỗi bước
                
    return None

In [63]:
def depth_first_search_2(initial_state, gui):
    node = Node(initial_state)

    if goal_test(node.state):
        return solution(node)

    frontier = deque()
    frontier.append(node)
    explored = []

    while len(frontier) > 0:
        node = frontier.pop() # Lấy node ở cuối frontier ra (Stack: LIFO)
        explored.append(node.state)

        for action in actions(node.state):
            child = child_node(node, action)

            in_explored = False
            for s in explored:
                if s == child.state:
                    in_explored = True

            in_frontier = False
            for n in frontier:
                if n.state == child.state:
                    in_frontier = True

            if not in_explored and not in_frontier:
                if goal_test(child.state):
                    return solution(child)

                frontier.append(child)
                
        gui.log_step(node, frontier, explored, "Explored") # Gọi hàm log_step để cập nhật giao diện sau mỗi bước
                
    return None

In [64]:
def depth_limited_search_1(initial_state, limit, gui):
    node = Node(initial_state)
    
    if goal_test(node.state):
        return solution(node)

    frontier = deque()
    frontier.append(node)
    reached = []
    cutoff_occurred = False

    while len(frontier) > 0:
        node = frontier.pop() # Lấy node ở cuối frontier ra (Stack: LIFO)
        reached.append(node.state)

        if goal_test(node.state):
            return solution(node)

        # Nếu đạt đến giới hạn độ sâu, không mở rộng (expand) node này nữa
        if node.cost >= limit:
            cutoff_occurred = True
            gui.log_step(node, frontier, reached, "Reached")
            continue

        for action in actions(node.state):
            child = child_node(node, action)
            
            in_reached = False
            for s in reached:
                if s == child.state:
                    in_reached = True

            in_frontier = False
            for n in frontier:
                if n.state == child.state:
                    in_frontier = True

            if not in_reached and not in_frontier:
                frontier.append(child)
                
        gui.log_step(node, frontier, reached, "Reached")
                
    return "CUTOFF" if cutoff_occurred else None

In [65]:
def depth_limited_search_2(initial_state, limit, gui):
    node = Node(initial_state)

    if goal_test(node.state):
        return solution(node)

    frontier = deque()
    frontier.append(node)
    explored = []
    cutoff_occurred = False

    while len(frontier) > 0:
        node = frontier.pop() # Lấy node ở cuối frontier ra (Stack: LIFO)
        explored.append(node.state)

        # Nếu đạt đến giới hạn độ sâu, không mở rộng (expand) node này nữa
        if node.cost >= limit:
            cutoff_occurred = True
            gui.log_step(node, frontier, explored, "Explored")
            continue

        for action in actions(node.state):
            child = child_node(node, action)

            in_explored = False
            for s in explored:
                if s == child.state:
                    in_explored = True

            in_frontier = False
            for n in frontier:
                if n.state == child.state:
                    in_frontier = True

            if not in_explored and not in_frontier:
                if goal_test(child.state):
                    return solution(child)

                frontier.append(child)
                
        gui.log_step(node, frontier, explored, "Explored")
                
    return "CUTOFF" if cutoff_occurred else None

In [66]:
def iterative_deepening_search_1(initial_state, gui):
    max_depth = 50 # Giới hạn độ sâu an toàn để tránh lặp vô hạn
    
    for depth in range(max_depth):
        # Thông báo ra log GUI cho dễ theo dõi
        gui.log_text.insert(tk.END, f"\n{'*'*14} Depth = {depth} {'*'*14}\n")
        
        # Reset lại Node counter và log hiển thị để bắt đầu vòng lặp mới sạch sẽ
        Node.reset_counter()
        if hasattr(gui, 'explored_names_log'):
            gui.explored_names_log = []

        result = depth_limited_search_1(initial_state, depth, gui)
        
        # Nếu tìm thấy kết quả hợp lệ (không phải bị cutoff và không phải thất bại)
        if result != "CUTOFF" and result is not None:
            return result
        
        # Nếu duyệt hết toàn bộ không gian trạng thái mà không bị cutoff -> Thất bại
        if result is None:
            return None
            
    return None

In [67]:
def iterative_deepening_search_2(initial_state, gui):
    max_depth = 50 
    
    for depth in range(max_depth):
        gui.log_text.insert(tk.END, f"\n{'*'*14} Depth = {depth} {'*'*14}\n")
        
        Node.reset_counter()
        if hasattr(gui, 'explored_names_log'):
            gui.explored_names_log = []

        result = depth_limited_search_2(initial_state, depth, gui)
        
        if result != "CUTOFF" and result is not None:
            return result
            
        if result is None:
            return None
            
    return None

In [68]:
def uniform_cost_search(initial_state, gui):
    node = Node(initial_state)
    
    # Ghi đè cost của root bằng số lượng bụi hiện tại
    node.cost = count_dust(node.state) 
    
    if goal_test(node.state):
        return solution(node)

    # Priority Queue: Lưu tuple (cost, name, node) để heap tự động sắp xếp theo cost
    # Thuộc tính name được đưa vào để tie-break (tránh lỗi khi 2 node có cùng cost)
    frontier = []
    heapq.heappush(frontier, (node.cost, node.name, node))
    reached = []

    while len(frontier) > 0:
        # Lấy node có cost nhỏ nhất ra khỏi Priority Queue
        current_cost, _, node = heapq.heappop(frontier)
        reached.append(node.state)

        if goal_test(node.state):
            return solution(node)

        for action in actions(node.state):
            child = child_node(node, action)
            
            # Tính Path Cost mới: Cost cha + Số lượng bụi của Node con (số ô sai so với GOAL)
            child.cost = node.cost + count_dust(child.state)

            in_reached = False
            for s in reached:
                if s == child.state:
                    in_reached = True

            in_frontier = False
            for i, (f_cost, f_name, f_node) in enumerate(frontier):
                if f_node.state == child.state:
                    in_frontier = True
                    # Nếu trạng thái đã có trong Frontier nhưng chi phí mới tốt hơn -> Thay thế node cũ
                    if child.cost < f_cost:
                        frontier[i] = (child.cost, child.name, child)
                        heapq.heapify(frontier) # Cấu trúc lại cấu trúc heap sau khi thay đổi
                    break

            if not in_reached and not in_frontier:
                heapq.heappush(frontier, (child.cost, child.name, child))
                
        # Format lại frontier (chỉ lấy mảng node) để đưa vào hàm log_step in ra giao diện
        frontier_nodes = [item[2] for item in frontier]
        gui.log_step(node, frontier_nodes, reached, "Reached")
        
    return None

In [69]:
def greedy_search(initial_state, gui):
    node = Node(initial_state)
    
    # h(n) = số lượng bụi hiện tại (số ô sai so với trạng thái đích)
    node.cost = count_dust(node.state) 
    
    if goal_test(node.state):
        return solution(node)

    # Priority Queue: Lưu tuple (h(n), name, node) để heap tự động sắp xếp theo h(n)
    frontier = []
    heapq.heappush(frontier, (node.cost, node.name, node))
    reached = []

    while len(frontier) > 0:
        # Lấy node có cost h(n) nhỏ nhất ra khỏi Priority Queue
        current_cost, _, node = heapq.heappop(frontier)
        reached.append(node.state)

        if goal_test(node.state):
            return solution(node)

        for action in actions(node.state):
            child = child_node(node, action)
            
            # Cost chỉ là h(n) của node hiện tại đang xét, KHÔNG cộng dồn cost của node cha
            child.cost = count_dust(child.state)

            in_reached = False
            for s in reached:
                if s == child.state:
                    in_reached = True

            in_frontier = False
            for f_cost, f_name, f_node in frontier:
                if f_node.state == child.state:
                    in_frontier = True
                    break

            if not in_reached and not in_frontier:
                heapq.heappush(frontier, (child.cost, child.name, child))
                
        # Format lại frontier để đưa vào hàm log_step in ra giao diện
        frontier_nodes = [item[2] for item in frontier]
        gui.log_step(node, frontier_nodes, reached, "Reached")
        
    return None

In [70]:
def get_dust_positions(matrix):
    dusts = []
    for i in range(len(matrix)):
        for j in range(len(matrix[0])):
            if matrix[i][j] == 1:
                dusts.append((i, j))
    return dusts

def heuristic_cost(state):
    robot_pos = find_robot(state)
    dusts = get_dust_positions(state)
    
    if not dusts or not robot_pos:
        return 0
        
    rx, ry = robot_pos
    min_distance = float('inf')
    
    # Tìm khoảng cách Manhattan đến hạt bụi gần nhất
    for dx, dy in dusts:
        distance = abs(rx - dx) + abs(ry - dy)
        if distance < min_distance:
            min_distance = distance
            
    # h(n): Manhattan gần nhất + (tổng số bụi - 1)
    return min_distance + (len(dusts) - 1)

In [71]:
def a_star_search(initial_state, gui):
    node = Node(initial_state)
    
    # g(n): Số ô sai (số lượng bụi), có kế thừa (tích lũy)
    node.g = count_dust(node.state)
    # h(n): Manhattan gần nhất + (tổng số bụi - 1), không kế thừa
    node.h = heuristic_cost(node.state)
    # f(n) = g(n) + h(n). Gán vào node.cost để dùng chung hàm log_step
    node.cost = node.g + node.h 
    
    if goal_test(node.state):
        return solution(node)

    # Priority Queue: Lưu tuple (f(n), name, node) để tự động sắp xếp theo f(n)
    frontier = []
    heapq.heappush(frontier, (node.cost, node.name, node))
    reached = []

    while len(frontier) > 0:
        # Lấy node có f(n) nhỏ nhất ra khỏi Priority Queue
        current_cost, _, node = heapq.heappop(frontier)
        reached.append(node.state)

        if goal_test(node.state):
            return solution(node)

        for action in actions(node.state):
            child = child_node(node, action)
            
            child.g = node.g + count_dust(child.state)
            
            child.h = heuristic_cost(child.state)
            
            # Tính f(n)
            child.cost = child.g + child.h

            in_reached = False
            for s in reached:
                if s == child.state:
                    in_reached = True
                    break

            in_frontier = False
            for i, (f_cost, f_name, f_node) in enumerate(frontier):
                if f_node.state == child.state:
                    in_frontier = True
                    # Nếu trạng thái đã có trong Frontier nhưng chi phí mới tốt hơn -> Thay thế node cũ
                    if child.cost < f_cost:
                        frontier[i] = (child.cost, child.name, child)
                        heapq.heapify(frontier) # Cấu trúc lại cấu trúc heap sau khi thay đổi
                    break

            if not in_reached and not in_frontier:
                heapq.heappush(frontier, (child.cost, child.name, child))
                
        # Format lại frontier để đưa vào hàm log_step in ra giao diện
        frontier_nodes = [item[2] for item in frontier]
        gui.log_step(node, frontier_nodes, reached, "Reached")
        
    return None

In [72]:
class CleaningRobotGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Cleaning Robot")
        self.root.geometry("950x600")
        self.root.configure(bg="#F5F5F5")

        self.style = ttk.Style()
        self.style.theme_use('clam')
        self.style.configure('TButton', font=('Segoe UI', 10), padding=5)
        self.style.configure('TLabel', font=('Segoe UI', 10), background="#F5F5F5")

        self.initial_matrix = []
        self.current_matrix = []
        self.is_running = False
        self.canvas_dim = 360 # Cố định kích thước khung vẽ ma trận

        self.setup_ui()
        self.generate_map()

    def setup_ui(self):
        # Khu vực phía trên (Chứa Controls, Canvas, Log)
        self.top_frame = tk.Frame(self.root, bg="#F5F5F5")
        self.top_frame.pack(side=tk.TOP, fill=tk.BOTH, expand=True, padx=20, pady=20)

        # Cột 1: Controls (Nút bấm & Tùy chọn)
        self.control_frame = tk.Frame(self.top_frame, bg="#F5F5F5")
        self.control_frame.pack(side=tk.LEFT, fill=tk.Y, padx=(0, 20))

        ttk.Label(self.control_frame, text="Algorithms", font=('Segoe UI', 12, 'bold')).pack(pady=(0, 10))
        ttk.Button(self.control_frame, text="BFS 1", command=lambda: self.run_algo("BFS1"), width=15).pack(pady=5)
        ttk.Button(self.control_frame, text="BFS 2", command=lambda: self.run_algo("BFS2"), width=15).pack(pady=5)
        ttk.Button(self.control_frame, text="DFS 1", command=lambda: self.run_algo("DFS1"), width=15).pack(pady=5)
        ttk.Button(self.control_frame, text="DFS 2", command=lambda: self.run_algo("DFS2"), width=15).pack(pady=5)
        ttk.Button(self.control_frame, text="IDS 1", command=lambda: self.run_algo("IDS1"), width=15).pack(pady=5)
        ttk.Button(self.control_frame, text="IDS 2", command=lambda: self.run_algo("IDS2"), width=15).pack(pady=5)
        ttk.Button(self.control_frame, text="UCS", command=lambda: self.run_algo("UCS"), width=15).pack(pady=5)
        ttk.Button(self.control_frame, text="Greedy Search", command=lambda: self.run_algo("GS"), width=15).pack(pady=5)
        ttk.Button(self.control_frame, text="A* Search", command=lambda: self.run_algo("A*"), width=15).pack(pady=5)

        ttk.Separator(self.control_frame, orient='horizontal').pack(fill='x', pady=15)
        ttk.Button(self.control_frame, text="Generate Map", command=self.generate_map, width=15).pack(pady=20)

        # Cột 2: Canvas (Ma trận)
        self.canvas_frame = tk.Frame(self.top_frame, bg="white", bd=1, relief="solid")
        self.canvas_frame.pack(side=tk.LEFT, padx=10)
        self.canvas = tk.Canvas(self.canvas_frame, width=self.canvas_dim, height=self.canvas_dim, bg="white", highlightthickness=0)
        self.canvas.pack(padx=2, pady=2)

        # Cột 3: Log
        self.log_frame = tk.Frame(self.top_frame, bg="#F5F5F5")
        self.log_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=(20, 0))
        
        ttk.Label(self.log_frame, text="Search Execution Log", font=('Segoe UI', 10, 'bold')).pack(anchor="w", pady=(0, 5))
        self.log_text = scrolledtext.ScrolledText(self.log_frame, width=35, font=("Consolas", 9), bd=1, relief="solid")
        self.log_text.pack(fill=tk.BOTH, expand=True)

        # Khu vực phía dưới (Solution Box)
        self.bottom_frame = tk.Frame(self.root, bg="#F5F5F5")
        self.bottom_frame.pack(side=tk.BOTTOM, fill=tk.X, padx=20, pady=(0, 20))
        
        ttk.Label(self.bottom_frame, text="Solution Actions:", font=('Segoe UI', 10, 'bold')).pack(anchor="w", pady=(0, 5))
        self.sol_text = tk.Text(self.bottom_frame, height=3, font=("Consolas", 10), bd=1, relief="solid", bg="#FAFAFA")
        self.sol_text.pack(fill=tk.X)
        self.sol_text.config(state=tk.DISABLED)

    def generate_map(self):
        if self.is_running: return
        
        size = 3
        # 1. Khởi tạo toàn bộ ma trận là 0 (Sạch)
        self.initial_matrix = [[0 for _ in range(size)] for _ in range(size)]
        
        # 2. Đặt robot ngẫu nhiên (Giá trị 2)
        self.initial_matrix[random.randint(0, size-1)][random.randint(0, size-1)] = 2
        
        # 3. Đặt chướng ngại vật ngẫu nhiên (Giá trị 3)
        num_obstacles = 2
        obs_placed = 0
        while obs_placed < num_obstacles:
            rx, ry = random.randint(0, size-1), random.randint(0, size-1)
            if self.initial_matrix[rx][ry] == 0:
                self.initial_matrix[rx][ry] = 3
                obs_placed += 1
                
        # 4. Đặt bụi ngẫu nhiên (Giá trị 1)
        # Sinh số lượng bụi ngẫu nhiên (từ 1 đến toàn bộ số ô trên bản đồ)
        num_dust = random.randint(1, 6)
        dust_placed = 0
        while dust_placed < num_dust:
            rx, ry = random.randint(0, size-1), random.randint(0, size-1)
            if self.initial_matrix[rx][ry] == 0:
                self.initial_matrix[rx][ry] = 1
                dust_placed += 1
                
        self.current_matrix = [row[:] for row in self.initial_matrix]
        self.draw_grid(self.current_matrix)
        
        self.log_text.delete(1.0, tk.END)
        self.log_text.insert(tk.END, f"Generated map with {num_dust} dust particles.\nReady for search.\n")
        self.update_solution_text("")

    def draw_grid(self, matrix):
        self.canvas.delete("all")
        size = len(matrix)
        cell_size = self.canvas_dim / size
        
        colors = {0: "#A5D6A7", 1: "#E0E0E0", 2: "#64B5F6", 3: "#E57373"}
        labels = {0: "Clean", 1: "Dust", 2: "Robot", 3: "Wall"}
        
        for i in range(size):
            for j in range(size):
                val = matrix[i][j]
                x1, y1 = j * cell_size, i * cell_size
                x2, y2 = x1 + cell_size, y1 + cell_size
                
                self.canvas.create_rectangle(x1, y1, x2, y2, fill=colors[val], outline="#FFFFFF", width=2)
                
                # Cỡ chữ linh hoạt theo kích thước ma trận
                font_size = 12 if size <= 3 else max(8, int(12 - (size - 3) * 1.5))
                self.canvas.create_text(x1 + cell_size/2, y1 + cell_size/2, 
                                        text=labels[val], font=("Segoe UI", font_size, "bold"), fill="#424242" if val in [0, 1] else "#FFFFFF")
                
        self.root.update_idletasks()
        
    def state_to_str(self, state):
        res = []
        for row in state:
            # Giữ nguyên giá trị gốc của ma trận (chỉ ép kiểu về string để nối chuỗi)
            res.append("[" + ",".join(str(val) for val in row) + "]")
        return "[" + ",".join(res) + "]"
    
    def short_action(self, action):
        if not action: return "-"
        return {"UP": "U", "DOWN": "D", "LEFT": "L", "RIGHT": "R"}.get(action, action)

    def log_step(self, current_node, frontier, closed_list, closed_name):
        # Lưu trữ danh sách tên [A.State, B.State]
        if not hasattr(self, 'explored_names_log'):
            self.explored_names_log = []
            
        explored_val = current_node.name + ".State"
        if explored_val not in self.explored_names_log:
            self.explored_names_log.append(explored_val)

        # Định dạng Frontier: {[State], Parent, Action, Cost} NodeName
        frontier_strs = []
        for n in frontier:
            state_str = self.state_to_str(n.state)
            parent_name = n.parent.name if n.parent else "-"
            act = self.short_action(n.action)
            cost = n.cost
            frontier_strs.append(f"  {{{state_str}, {parent_name}, {act}, {cost}}} {n.name}")
        
        frontier_display = "[\n" + ",\n".join(frontier_strs) + "\n]" if frontier_strs else "[]"
        explored_display = "[" + ", ".join(self.explored_names_log) + "]"

        # Xuất ra Log
        log_msg =  f"Node     : {current_node.name}\n"
        log_msg += f"Frontier : {frontier_display}\n"
        log_msg += f"Explored : {explored_display}\n"
        log_msg += "-" * 40 + "\n"
        
        self.log_text.insert(tk.END, log_msg)
        self.log_text.see(tk.END)
        self.root.update()

    def update_solution_text(self, text):
        self.sol_text.config(state=tk.NORMAL)
        self.sol_text.delete(1.0, tk.END)
        self.sol_text.insert(tk.END, text)
        self.sol_text.config(state=tk.DISABLED)

    def run_algo(self, algo_name):
        if self.is_running: return
        self.is_running = True
        
        Node.reset_counter()
        self.explored_names_log = []
        
        self.current_matrix = [row[:] for row in self.initial_matrix]
        self.draw_grid(self.current_matrix)
        
        self.log_text.delete(1.0, tk.END)
        self.log_text.insert(tk.END, f"[{algo_name}] Execution Started\n{'='*40}\n")
        
        # In ra node khởi tạo (Initial Node) trước khi bắt đầu thuật toán
        init_state_str = self.state_to_str(self.current_matrix)
        # Xác định cost khởi tạo (UCS tính cost dựa trên số bụi, các thuật toán khác bắt đầu bằng 0)
        if algo_name in ["UCS", "GS"]:
            init_cost = count_dust(self.current_matrix)
        elif algo_name == "A*":
            init_g = count_dust(self.current_matrix)
            init_h = heuristic_cost(self.current_matrix) # Gọi hàm heuristic mới
            init_cost = init_g + init_h
        else:
            init_cost = 0
            
        self.log_text.insert(tk.END, f"Initial Node:\n  {{{init_state_str}, _, _, {init_cost}}} A\n")
        self.log_text.insert(tk.END, f"{'-'*40}\n")
        
        self.update_solution_text("Searching...")
        
        path = None
        if algo_name == "BFS1": path = breadth_first_search_1(self.current_matrix, self)
        elif algo_name == "BFS2": path = breadth_first_search_2(self.current_matrix, self)
        elif algo_name == "DFS1": path = depth_first_search_1(self.current_matrix, self)
        elif algo_name == "DFS2": path = depth_first_search_2(self.current_matrix, self)
        elif algo_name == "IDS1": path = iterative_deepening_search_1(self.current_matrix, self)
        elif algo_name == "IDS2": path = iterative_deepening_search_2(self.current_matrix, self)
        elif algo_name == "UCS": path = uniform_cost_search(self.current_matrix, self)
        elif algo_name == "GS": path = greedy_search(self.current_matrix, self)
        elif algo_name == "A*": path = a_star_search(self.current_matrix, self)

        if path is None or len(path) == 0:
            if count_dust(self.current_matrix) == 0:
                self.update_solution_text("Bản đồ đã sạch sẽ (Clean).")
            else:
                # Chương trình tự động dừng do đã cạn kiệt không gian trạng thái
                error_msg = "Không tìm thấy solution"
                
                # In thông báo ra màn hình GUI
                self.update_solution_text(error_msg)
                
                # In thông báo vào bảng Log
                self.log_text.insert(tk.END, f"\n[THÔNG BÁO] {error_msg}!\nRobot đã dừng hoạt động.\n")
                self.log_text.see(tk.END)
        else:
            self.update_solution_text(f"Total steps: {len(path)}\nPath: " + " -> ".join(path))
            self.animate_path(path)
            
        self.is_running = False

    def animate_path(self, path):
        self.log_text.insert(tk.END, f"\n[Executing Solution Path]\n{'='*32}\n")
        for idx, action in enumerate(path):
            self.log_text.insert(tk.END, f"Step {idx+1:02d}: {action}\n")
            self.log_text.see(tk.END)
            
            self.current_matrix = result(self.current_matrix, action)
            self.draw_grid(self.current_matrix)
            
            self.root.update()
            time.sleep(0.3)
        
        self.log_text.insert(tk.END, "\nAll dust cleaned.")
        self.log_text.see(tk.END)

In [73]:
# Chạy chương trình
if __name__ == "__main__":
    root = tk.Tk()
    app = CleaningRobotGUI(root)
    root.mainloop()